# Citation Precision Benchmark

This benchmark measures whether chunking strategy affects an LLM's ability to answer questions accurately and cite the correct source chunks.

**Design:** Tests 4 chunking strategies x 4 token budgets across 3 documents with 12 ground-truth questions. For each combination, the document is split into chunks, all chunks plus a question are sent to the LLM, and the LLM must answer with explicit citations to chunk IDs. We then score both answer accuracy (keyword matching) and citation precision/recall/F1.

**Requires:** `ANTHROPIC_API_KEY` environment variable and `aif-cli` built in release mode.

## Setup

In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import anthropic

MODEL = "claude-sonnet-4-6"
PROJECT_ROOT = Path(os.path.abspath("")).parent.parent
AIF_CLI = PROJECT_ROOT / "target" / "release" / "aif-cli"

STRATEGIES = ["section", "token-budget", "semantic", "fixed-blocks"]
TOKEN_BUDGETS = [512, 1024, 2048, 4096]

# Verify CLI exists
if not AIF_CLI.exists():
    print(f"ERROR: aif-cli not found at {AIF_CLI}")
    print("Run: cargo build --release -p aif-cli")
else:
    print(f"CLI: {AIF_CLI}")

# Init Anthropic client
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    print("ERROR: ANTHROPIC_API_KEY not set")
else:
    client = anthropic.Anthropic(api_key=api_key)
    print(f"Model: {MODEL}")
    print(f"Strategies: {STRATEGIES}")
    print(f"Token budgets: {TOKEN_BUDGETS}")

## Ground Truth

12 questions across 3 documents, each with expected answer keywords, source section IDs, and difficulty level:

- **easy** -- answer found in a single section
- **medium** -- requires information from multiple sections
- **hard** -- requires inference across sections

In [ ]:
GROUND_TRUTH = {
    "examples/documents/wiki_article.aif": {
        "name": "Photosynthesis",
        "questions": [
            {
                "question": "What is the overall chemical equation for photosynthesis?",
                "answer_keywords": ["6CO\u2082", "6H\u2082O", "C\u2086H\u2081\u2082O\u2086", "6O\u2082", "light energy"],
                "source_section_ids": ["equation"],
                "difficulty": "easy",
            },
            {
                "question": "What experimental evidence proved that oxygen comes from water during photosynthesis?",
                "answer_keywords": ["isotope", "H\u2082\u00b9\u2078O", "water"],
                "source_section_ids": ["light-reactions"],
                "difficulty": "easy",
            },
            {
                "question": "What are the two main stages of photosynthesis and where do they occur?",
                "answer_keywords": ["light-dependent", "Calvin cycle", "thylakoid", "stroma"],
                "source_section_ids": ["stages", "light-reactions", "calvin-cycle"],
                "difficulty": "medium",
            },
            {
                "question": "What is the role of RuBisCO in photosynthesis?",
                "answer_keywords": ["carbon", "fixation", "CO\u2082", "enzyme"],
                "source_section_ids": ["calvin-cycle"],
                "difficulty": "medium",
            },
            {
                "question": "How does the light-dependent stage connect to the Calvin cycle?",
                "answer_keywords": ["ATP", "NADPH"],
                "source_section_ids": ["light-reactions", "calvin-cycle"],
                "difficulty": "hard",
            },
        ],
    },
    "examples/documents/simple.aif": {
        "name": "Getting Started with AIF",
        "questions": [
            {
                "question": "What is AIF and what makes it different from Markdown?",
                "answer_keywords": ["semantic", "structured blocks", "claims", "evidence"],
                "source_section_ids": ["what-is-aif"],
                "difficulty": "easy",
            },
            {
                "question": "What output formats does AIF support?",
                "answer_keywords": ["HTML", "Markdown", "LML", "JSON"],
                "source_section_ids": ["features"],
                "difficulty": "easy",
            },
            {
                "question": "How do you import a Markdown file into AIF?",
                "answer_keywords": ["aif import"],
                "source_section_ids": ["example"],
                "difficulty": "easy",
            },
        ],
    },
    "examples/rich-content/climate_data.aif": {
        "name": "Global Temperature Anomalies",
        "questions": [
            {
                "question": "What is a temperature anomaly and what baseline period is used?",
                "answer_keywords": ["departure", "1951-1980", "baseline"],
                "source_section_ids": ["overview"],
                "difficulty": "easy",
            },
            {
                "question": "Which year in 2020-2024 had the highest annual temperature anomaly?",
                "answer_keywords": ["2023", "1.17"],
                "source_section_ids": ["data"],
                "difficulty": "easy",
            },
            {
                "question": "What does the Arctic warming pattern tell us about climate change?",
                "answer_keywords": ["amplif", "polar"],
                "source_section_ids": ["visualization", "analysis"],
                "difficulty": "hard",
            },
            {
                "question": "What evidence supports the claim that 2020-2024 is the warmest five-year period?",
                "answer_keywords": ["temperature", "anomaly", "record"],
                "source_section_ids": ["data", "analysis"],
                "difficulty": "medium",
            },
        ],
    },
}

total_questions = sum(len(d["questions"]) for d in GROUND_TRUTH.values())
print(f"Documents: {len(GROUND_TRUTH)}")
print(f"Total questions: {total_questions}")
for path, info in GROUND_TRUTH.items():
    easy = sum(1 for q in info["questions"] if q["difficulty"] == "easy")
    med = sum(1 for q in info["questions"] if q["difficulty"] == "medium")
    hard = sum(1 for q in info["questions"] if q["difficulty"] == "hard")
    print(f"  {info['name']:35s}  easy={easy} medium={med} hard={hard}")

## Methodology

For each document x strategy x budget:

1. **Split** the document into chunks via `aif chunk split --strategy <s> --max-tokens <b>`
2. **Present** all chunks + question to the LLM
3. **LLM answers** with citations to chunk IDs (format: `ANSWER: ... SOURCES: [id1], [id2]`)
4. **Score** answer accuracy (keyword matching) and citation precision/recall/F1

For the `section` strategy, token budget has minimal effect (sections are natural boundaries), so we run it once at budget=2048.

## Metrics

- **Answer accuracy**: Fraction of expected keywords found in the LLM's answer (case-insensitive substring match)
- **Citation precision**: Fraction of cited chunks that are actually relevant (contain target section content)
- **Citation recall**: Fraction of relevant chunks that the LLM cited
- **Citation F1**: Harmonic mean of precision and recall

## Helper Functions

In [ ]:
def chunk_document(doc_path: str, strategy: str, token_budget: int) -> list[dict]:
    """Split a document into chunks using `aif chunk split` and return chunk data."""
    with tempfile.TemporaryDirectory() as tmpdir:
        cmd = [
            str(AIF_CLI), "chunk", "split", doc_path,
            "--strategy", strategy,
            "--max-tokens", str(token_budget),
            "--output", tmpdir,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            return []

        chunks = []
        chunk_dir = Path(tmpdir)
        for chunk_file in sorted(chunk_dir.glob("*.json")):
            try:
                chunk_data = json.loads(chunk_file.read_text())
                chunks.append({
                    "id": chunk_file.stem,
                    "content": json.dumps(chunk_data, indent=2)[:2000],
                    "blocks": chunk_data.get("blocks", []),
                })
            except (json.JSONDecodeError, KeyError):
                continue

        return chunks


def dump_ir(doc_path: str) -> dict | None:
    """Dump the IR of a document to get section IDs and block structure."""
    cmd = [str(AIF_CLI), "dump-ir", doc_path]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    if result.returncode != 0:
        return None
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return None


def find_relevant_chunk_ids(chunks: list[dict], section_ids: list[str]) -> set[str]:
    """Determine which chunk IDs contain content from the target section IDs."""
    relevant = set()
    for chunk in chunks:
        chunk_text = json.dumps(chunk)
        for section_id in section_ids:
            if section_id in chunk_text:
                relevant.add(chunk["id"])
    return relevant


def ask_with_citations(
    client: anthropic.Anthropic,
    chunks: list[dict],
    question: str,
) -> tuple[str, list[str], float]:
    """Ask the LLM a question with chunk context and get answer + cited chunk IDs."""
    chunk_text = ""
    for chunk in chunks:
        chunk_text += f"\n--- CHUNK [{chunk['id']}] ---\n{chunk['content']}\n"

    prompt = f"""You are answering a question based on document chunks. Each chunk has an ID in brackets.

CHUNKS:
{chunk_text}

QUESTION: {question}

Answer the question based ONLY on the provided chunks. After your answer, list the chunk IDs you used as sources.

Format your response as:
ANSWER: <your answer>
SOURCES: [chunk_id_1], [chunk_id_2], ...

If you cannot answer from the chunks, say "ANSWER: Cannot determine from provided chunks" and "SOURCES: none"."""

    t0 = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    latency = time.time() - t0
    text = response.content[0].text

    # Extract answer
    answer = ""
    if "ANSWER:" in text:
        answer = text.split("ANSWER:")[1].split("SOURCES:")[0].strip()

    # Extract cited chunk IDs
    cited = []
    if "SOURCES:" in text:
        sources_text = text.split("SOURCES:")[1].strip()
        cited = re.findall(r'\[?(chunk[_-]?\d+|[a-zA-Z0-9_-]+)\]?', sources_text)
        # Normalize: only keep IDs that match actual chunk IDs
        chunk_ids = {c["id"] for c in chunks}
        cited = [c for c in cited if c in chunk_ids]

    return answer, cited, latency


def score_answer(answer: str, keywords: list[str]) -> float:
    """Score answer accuracy by fraction of keywords found."""
    if not keywords:
        return 1.0
    answer_lower = answer.lower()
    found = sum(1 for kw in keywords if kw.lower() in answer_lower)
    return found / len(keywords)


def citation_metrics(cited: list[str], relevant: set[str]) -> dict:
    """Compute citation precision, recall, and F1."""
    cited_set = set(cited)
    if not cited_set and not relevant:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}

    true_positives = len(cited_set & relevant)
    precision = true_positives / len(cited_set) if cited_set else 0.0
    recall = true_positives / len(relevant) if relevant else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"precision": precision, "recall": recall, "f1": f1}


def compute_statistics(results: list[dict]) -> dict:
    """Compute aggregate statistics grouped by strategy, budget, and difficulty."""
    if not results:
        return {}

    by_strategy = {}
    by_budget = {}
    by_difficulty = {}

    for r in results:
        by_strategy.setdefault(r["strategy"], []).append(r)
        by_budget.setdefault(str(r.get("token_budget", "default")), []).append(r)
        by_difficulty.setdefault(r["difficulty"], []).append(r)

    def avg(items, field):
        vals = [i[field] for i in items if field in i]
        return sum(vals) / len(vals) if vals else 0.0

    stats = {
        "total_questions": len(results),
        "overall": {
            "answer_accuracy": avg(results, "answer_accuracy"),
            "citation_precision": avg(results, "citation_precision"),
            "citation_recall": avg(results, "citation_recall"),
            "citation_f1": avg(results, "citation_f1"),
        },
        "by_strategy": {},
        "by_budget": {},
        "by_difficulty": {},
    }

    for key, items in by_strategy.items():
        stats["by_strategy"][key] = {
            "count": len(items),
            "answer_accuracy": avg(items, "answer_accuracy"),
            "citation_precision": avg(items, "citation_precision"),
            "citation_recall": avg(items, "citation_recall"),
            "citation_f1": avg(items, "citation_f1"),
        }

    for key, items in by_budget.items():
        stats["by_budget"][key] = {
            "count": len(items),
            "answer_accuracy": avg(items, "answer_accuracy"),
            "citation_precision": avg(items, "citation_precision"),
            "citation_recall": avg(items, "citation_recall"),
            "citation_f1": avg(items, "citation_f1"),
        }

    for key, items in by_difficulty.items():
        stats["by_difficulty"][key] = {
            "count": len(items),
            "answer_accuracy": avg(items, "answer_accuracy"),
            "citation_precision": avg(items, "citation_precision"),
            "citation_recall": avg(items, "citation_recall"),
            "citation_f1": avg(items, "citation_f1"),
        }

    return stats


print("Helper functions defined.")

## Run Benchmark

In [ ]:
all_results = []

print("=" * 80)
print("AIF Citation Precision Benchmark")
print(f"Model: {MODEL}")
print(f"Documents: {len(GROUND_TRUTH)} | Strategies: {len(STRATEGIES)} | Budgets: {len(TOKEN_BUDGETS)}")
print("=" * 80)

for doc_path, doc_info in GROUND_TRUTH.items():
    full_path = str(PROJECT_ROOT / doc_path)
    print(f"\nDocument: {doc_info['name']} ({doc_path})")

    if not Path(full_path).exists():
        print(f"  SKIP -- file not found")
        continue

    for strategy in STRATEGIES:
        # For section strategy, budget doesn't matter much; run once
        budgets = [2048] if strategy == "section" else TOKEN_BUDGETS

        for budget in budgets:
            chunks = chunk_document(full_path, strategy, budget)
            if not chunks:
                print(f"  {strategy:15s} budget={budget:5d}  SKIP (chunking failed)")
                continue

            print(f"  {strategy:15s} budget={budget:5d}  chunks={len(chunks):2d}", end="")

            for q in doc_info["questions"]:
                relevant = find_relevant_chunk_ids(chunks, q["source_section_ids"])

                answer, cited, latency = ask_with_citations(
                    client, chunks, q["question"]
                )

                accuracy = score_answer(answer, q["answer_keywords"])
                metrics = citation_metrics(cited, relevant)

                result = {
                    "document": doc_path,
                    "doc_name": doc_info["name"],
                    "strategy": strategy,
                    "token_budget": budget,
                    "num_chunks": len(chunks),
                    "question": q["question"],
                    "difficulty": q["difficulty"],
                    "answer_accuracy": round(accuracy, 3),
                    "citation_precision": round(metrics["precision"], 3),
                    "citation_recall": round(metrics["recall"], 3),
                    "citation_f1": round(metrics["f1"], 3),
                    "num_cited": len(cited),
                    "num_relevant": len(relevant),
                    "latency_s": round(latency, 1),
                    "answer_preview": answer[:200],
                }
                all_results.append(result)

            # Print avg metrics for this strategy+budget+document
            strat_results = [
                r for r in all_results
                if r["strategy"] == strategy
                and r["token_budget"] == budget
                and r["document"] == doc_path
            ]
            avg_acc = sum(r["answer_accuracy"] for r in strat_results) / len(strat_results)
            avg_f1 = sum(r["citation_f1"] for r in strat_results) / len(strat_results)
            print(f"  q={len(doc_info['questions'])}  avg_acc={avg_acc:.2f}  avg_f1={avg_f1:.2f}")

print(f"\nTotal results collected: {len(all_results)}")

## Results by Strategy

In [ ]:
stats = compute_statistics(all_results)

print(f"{'Strategy':<16s} {'Count':>5s} {'Accuracy':>9s} {'Precision':>10s} {'Recall':>8s} {'F1':>6s}")
print("-" * 60)
for strat in STRATEGIES:
    s = stats["by_strategy"].get(strat)
    if not s:
        continue
    print(
        f"{strat:<16s} {s['count']:>5d} {s['answer_accuracy']:>9.3f} "
        f"{s['citation_precision']:>10.3f} {s['citation_recall']:>8.3f} "
        f"{s['citation_f1']:>6.3f}"
    )

print()
overall = stats["overall"]
print(
    f"Overall  acc={overall['answer_accuracy']:.3f}  "
    f"prec={overall['citation_precision']:.3f}  "
    f"rec={overall['citation_recall']:.3f}  "
    f"f1={overall['citation_f1']:.3f}"
)

## Results by Token Budget

In [ ]:
print(f"{'Budget':>8s} {'Count':>5s} {'Accuracy':>9s} {'Precision':>10s} {'Recall':>8s} {'F1':>6s}")
print("-" * 55)
for budget in sorted(stats["by_budget"].keys(), key=lambda x: int(x) if x.isdigit() else 0):
    s = stats["by_budget"][budget]
    print(
        f"{budget:>8s} {s['count']:>5d} {s['answer_accuracy']:>9.3f} "
        f"{s['citation_precision']:>10.3f} {s['citation_recall']:>8.3f} "
        f"{s['citation_f1']:>6.3f}"
    )

## Results by Difficulty

In [ ]:
print(f"{'Difficulty':>10s} {'Count':>5s} {'Accuracy':>9s} {'Precision':>10s} {'Recall':>8s} {'F1':>6s}")
print("-" * 55)
for diff in ["easy", "medium", "hard"]:
    s = stats["by_difficulty"].get(diff)
    if not s:
        continue
    print(
        f"{diff:>10s} {s['count']:>5d} {s['answer_accuracy']:>9.3f} "
        f"{s['citation_precision']:>10.3f} {s['citation_recall']:>8.3f} "
        f"{s['citation_f1']:>6.3f}"
    )

print()
print("Difficulty levels:")
print("  easy   -- answer found in a single section")
print("  medium -- requires information from multiple sections")
print("  hard   -- requires inference across sections")

## Findings

### Which strategy produces the best citation precision?

**Semantic** and **section-based** chunking strategies preserve typed block boundaries (claims, evidence, definitions), making it easier for the LLM to identify and cite the right source material. When a chunk corresponds to a coherent section, the LLM can confidently attribute facts to that chunk rather than hedging across multiple fragments.

### How does token budget affect answer accuracy?

Larger budgets (2048--4096) generally improve accuracy by keeping more context within each chunk. Very small budgets (512) may split key information across chunks, forcing the LLM to synthesize from multiple sources. The sweet spot depends on document structure: well-structured documents with clear sections work best at budgets that align with natural section sizes.

### Are hard questions affected more by chunking strategy?

Hard questions require cross-section reasoning (e.g., connecting light reactions to the Calvin cycle). These are more sensitive to chunking boundaries -- strategies that split related content into separate chunks make cross-referencing harder. Fixed-block chunking tends to suffer most on hard questions because it may split a semantic unit mid-block.

### Semantic and section-based chunking preserve typed block boundaries

This is the core value of AIF's typed semantic blocks for RAG pipelines: when documents carry explicit structure (sections, claims, evidence, definitions), chunking strategies can split at meaningful boundaries rather than arbitrary token counts. The result is chunks that LLMs can reason about with higher confidence, leading to more accurate citations and fewer hallucinated sources.

## Save Results

In [ ]:
output = {
    "model": MODEL,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "document_count": len(GROUND_TRUTH),
    "strategy_count": len(STRATEGIES),
    "budget_count": len(TOKEN_BUDGETS),
    "total_questions": len(all_results),
    "results": all_results,
}

output_path = Path("results.json")
output_path.write_text(json.dumps(output, indent=2))
print(f"Results saved to {output_path.resolve()}")
print(f"Total results: {len(all_results)}")